In [1]:
import numpy as np
from sklearn.feature_selection import f_regression
import statsmodels.api as sm


In [2]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(float)
train_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(float)
validation_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(float)
test_targets = npz['outputs'].astype(int)




In [3]:
from sklearn.feature_selection import f_classif
import pandas as pd

# Compute F-scores and p-values
f_values, p_values = f_classif(train_inputs, train_targets)

# Put results in a table
feature_summary = pd.DataFrame({
    'Feature': np.arange(train_inputs.shape[1]),
    'F_value': f_values,
    'p_value': p_values
})

# Round for readability
feature_summary = feature_summary.round(3)
feature_summary

,Feature,F_value,p_value
0,0,0.027,0.870
1,1,379.533,0.000
2,2,12.471,0.000
3,3,63.472,0.000
4,4,4.047,0.044
5,5,1.635,0.201
6,6,799.348,0.000
7,7,756.972,0.000
8,8,6.252,0.012
9,9,0.001,0.974


In [4]:
col_to_delete = []
for i in range(len(feature_summary['p_value'])):
    if feature_summary['p_value'][i] >0.05:
        col_to_delete.append(i)

In [5]:
train_inputs = np.delete(train_inputs, col_to_delete, axis=1)
validation_inputs = np.delete(validation_inputs, col_to_delete, axis=1)
test_inputs = np.delete(test_inputs, col_to_delete, axis=1)

In [6]:
print(np.mean(train_inputs, axis=0))
print(np.std(train_inputs, axis=0))

[-0.00120725  0.00711603  0.00150982  0.00680856  0.0054401   0.00778617
  0.00943076]
[0.98753302 1.03079181 1.01994847 1.00552203 1.00134731 1.00400787
 1.04533453]


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, penalty='l1', solver='liblinear')
model.fit(train_inputs, train_targets)
print(model.score(train_inputs, train_targets))

0.7865325509918972


In [8]:
y_pred_train = model.predict(train_inputs)
y_pred_val = model.predict(validation_inputs)
y_pred_test = model.predict(test_inputs)

In [9]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


cm = confusion_matrix(train_targets, y_pred_train)
precision = precision_score(train_targets, y_pred_train)
recall = recall_score(train_targets, y_pred_train)
f1 = f1_score(train_targets, y_pred_train)

print("Confusion Matrix:\n", cm)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix:
 [[1085  721]
 [  43 1730]]
Precision: 0.7058343533251734
Recall: 0.975747320924986
F1 Score: 0.8191287878787878


In [10]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

accuracy = model.score(validation_inputs, validation_targets)
cm = confusion_matrix(validation_targets, y_pred_val)
precision = precision_score(validation_targets, y_pred_val)
recall = recall_score(validation_targets, y_pred_val)
f1 = f1_score(validation_targets, y_pred_val)

print("Confusion Matrix:\n", cm)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix:
 [[122  90]
 [  5 230]]
Precision: 0.71875
Recall: 0.9787234042553191
F1 Score: 0.8288288288288288


In [11]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

accuracy = model.score(test_inputs, test_targets)
cm = confusion_matrix(test_targets, y_pred_test)
precision = precision_score(test_targets, y_pred_test)
recall = recall_score(test_targets, y_pred_test)
f1 = f1_score(test_targets, y_pred_test)

print("Confusion Matrix:\n", cm)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix:
 [[130  89]
 [  6 223]]
Precision: 0.7147435897435898
Recall: 0.9737991266375546
F1 Score: 0.8243992606284658
